# From MF to Mamba4Rec · Run and verify the four pilots

Each session trains **one model** and scores October validation queries. Start with the default MF smoke run. Continue with LightGBM, LRURec and Mamba4Rec in that order. The smoke setting exercises training, ranking and checkpoint reload on a small sample. Select **baseline** for a full fixed-configuration run. Compare quality across models using runs on the same full validation data. The November test opens after model selection is frozen.

| Stage | Model | Colab runtime |
|---|---|---|
| D3 | MF | CPU |
| D4 | LightGBM | CPU |
| D5a | LRURec | NVIDIA GPU |
| D5b | Mamba4Rec with native selective scan | NVIDIA GPU with a supported CUDA driver |

Open **Runtime → Change runtime type** and choose the accelerator shown above. Run all cells in order. Wait for the final verification, then save the displayed run folder location. To move to another model, change the selection in the first code cell and rerun the notebook from the start. Changing the Colab runtime clears its temporary files. A new session reopens the frozen D2 data and writes a new run to the same Google Drive project. [Monthly data and evaluation](monthly_queries_and_evaluation.ipynb)

# 从 MF 到 Mamba4Rec · 运行并核验四个模型

每个会话训练**一个模型**，并对十月验证查询评分。先跑默认的 MF 小规模试跑，再依次跑 LightGBM、LRURec 和 Mamba4Rec。**smoke** 使用小样本验证从训练、排序到重载模型的完整过程；选 **baseline** 可用完整数据运行已固定参数的基线。比较模型效果时，请使用同一完整验证集的运行记录。十一月测试在模型选择冻结后开启。

| 阶段 | 模型 | Colab 运行时 |
|---|---|---|
| D3 | MF | CPU |
| D4 | LightGBM | CPU |
| D5a | LRURec | NVIDIA GPU |
| D5b | 使用原生选择性扫描算子的 Mamba4Rec | 驱动支持相应 CUDA 版本的 NVIDIA GPU |

打开 **Runtime → Change runtime type（更改运行时类型）**，按表选择硬件，再顺序运行全部单元格。等最后的核验完成，记下显示的运行文件夹。切换到下一个模型时，修改第一个代码单元格的选项，从头运行。更换 Colab 运行时会清空临时文件；新会话会重新读取同一份 D2 数据，并将本次记录写入同一个 Google Drive 项目。[月度数据与评估](monthly_queries_and_evaluation.ipynb)

In [1]:
#@title Choose the model, run size and D2 data / 选择模型、运行规模与 D2 数据
MODEL = "mf" #@param ["mf", "lightgbm", "lrurec", "mamba4rec"]
RUN_SIZE = "smoke" #@param ["smoke", "baseline"]
D2_SOURCE = "auto" #@param ["auto", "checkout", "drive"]
DRIVE_FOLDER = "CS5481_Project2_OnlineRetail" #@param {type:"string"}

DEVICES = {"mf": "cpu", "lightgbm": "cpu", "lrurec": "cuda", "mamba4rec": "cuda"}
if MODEL not in DEVICES or RUN_SIZE not in {"smoke", "baseline"} or D2_SOURCE not in {"auto", "checkout", "drive"}:
    raise ValueError("Choose one of the listed model, run size and data options.")
device = DEVICES[MODEL]
print(f"{MODEL} | {RUN_SIZE} | {device} | D2: {D2_SOURCE}")

mf | smoke | cpu | D2: auto


## Open the code, Drive and run log

In Colab, authorize Drive when prompted. **auto** checks for the D2 manifest in `My Drive/CS5481_Project2_OnlineRetail/data/processed/d2/` and uses the pinned GitHub data if the manifest is absent or the files differ. The log records the selected source. **checkout** always uses the committed data. **drive** requires the completed D2 outputs in the selected project folder. The next cell verifies all input hashes before training. The original Excel file is needed only when rebuilding D1/D2. [D2 notebook](monthly_queries_and_evaluation.ipynb)

Each run gets its own folder under `experiments/model_pilots/`. The console transcript is written there as the setup and training run. Local Jupyter uses the checkout's Git history to load the same code revision and can save to another folder via `RETAIL_PILOT_RUNS`.

## 打开代码、Drive 和运行日志

在 Colab 弹出提示时授权挂载 Drive。**auto** 会检查 `我的云端硬盘/CS5481_Project2_OnlineRetail/data/processed/d2/` 是否已有 D2 清单；如果清单缺失或文件校验不同，就自动改用固定 GitHub 版本中的 D2 数据，日志会说明实际来源。**checkout** 总是使用仓库数据，**drive** 要求所选 Drive 项目目录已包含完整的 D2 输出。下个单元格会在训练前核对所有输入文件的校验值。只有从头重建 D1/D2 时才需要原始 Excel。[D2 notebook](monthly_queries_and_evaluation.ipynb)

每次运行都会在 `experiments/model_pilots/` 下建立独立目录，实时写入安装和训练的终端日志。本地 Jupyter 从仓库的 Git 历史读取同一代码版本，也可以通过环境变量 `RETAIL_PILOT_RUNS` 指定结果位置。

In [2]:
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import platform
import shlex
import subprocess
import sys
import tempfile
import uuid

try:
    from google.colab import drive
except ImportError:
    drive = None
on_colab = drive is not None
runtime = "Google Colab" if on_colab else "local Jupyter"
revision = "df5e9278444acd71cca258d4ba66cb773810c180"
github = "https://github.com/NobunagaK/online-retail-top10-recommendation.git"
if on_colab:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    source_url = github
    cache = Path("/content/retail_model_code")
else:
    start = Path(os.environ.get("RETAIL_SOURCE_ROOT", Path.cwd())).resolve()
    project = next((p for p in [start, *start.parents] if (p / ".git").exists()), None)
    if project is None:
        raise RuntimeError("Open this notebook in the project checkout or set RETAIL_SOURCE_ROOT.")
    source_url = str(project)
    cache = Path(tempfile.gettempdir()) / "retail_model_code"
checkout = cache / revision
checkout.mkdir(parents=True, exist_ok=True)

def git(*args):
    return subprocess.check_output(["git", "-C", str(checkout), *args], text=True).strip()

if not (checkout / ".git").is_dir():
    git("init", "--quiet")
if subprocess.run(["git", "-C", str(checkout), "cat-file", "-e", revision + "^{commit}"],
                  capture_output=True).returncode:
    git("fetch", "--quiet", "--depth=1", source_url, revision)
git("-c", "advice.detachedHead=false", "checkout", "--quiet", "--detach", revision)
if git("rev-parse", "HEAD") != revision or git("status", "--porcelain", "--untracked-files=no"):
    raise RuntimeError("The source checkout differs from the pinned commit. Remove the temporary checkout and retry.")

runs = Path(os.environ.get("RETAIL_PILOT_RUNS", project / "experiments/model_pilots"))
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
session = runs / f"{stamp}_{MODEL}_{RUN_SIZE}_{uuid.uuid4().hex[:8]}"
session.mkdir(parents=True, exist_ok=False)
run_dir = session / "model"
transcript = session / "session.log"

def log(message):
    print(message, flush=True)
    with transcript.open("a", encoding="utf-8") as out:
        print(message, file=out)

def run(command, *, label=None):
    command = [str(x) for x in command]
    log(label or "$ " + shlex.join(command))
    with subprocess.Popen(command, cwd=checkout, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        with transcript.open("a", encoding="utf-8") as out:
            for line in process.stdout:
                print(line, end="", flush=True)
                out.write(line)
                out.flush()
        if process.wait() != 0:
            raise RuntimeError(f"Command failed (exit {process.returncode}). See {transcript}.")

reference = checkout / "data/processed/d2"
drive_data = project / "data/processed/d2"
if D2_SOURCE == "drive" and not on_colab:
    raise ValueError("Use auto or checkout in local Jupyter.")
reference_manifest = json.loads((reference / "manifest.json").read_text())

def verify_d2(folder):
    selected = json.loads((folder / "manifest.json").read_text())
    if selected["outputs"] != reference_manifest["outputs"] or selected["splits"] != ["train", "validation"]:
        raise ValueError("The D2 manifest differs from the pinned training/validation manifest.")
    for name, record in reference_manifest["outputs"].items():
        with (folder / name).open("rb") as src:
            if hashlib.file_digest(src, "sha256").hexdigest() != record["sha256"]:
                raise ValueError(f"{name} differs from the pinned SHA-256.")
    return selected

log(f"Runtime: {runtime}")
log(f"Session: {session}")
log(f"Source: {revision}")
data = drive_data if on_colab and D2_SOURCE != "checkout" else reference
try:
    selected = verify_d2(data)
except (OSError, KeyError, ValueError, TypeError) as error:
    if data == reference:
        raise
    if D2_SOURCE == "drive":
        raise RuntimeError(f"Drive D2 verification failed: {error}. Select checkout or rerun the D2 notebook.") from error
    log(f"Drive D2 unavailable or different: {error}")
    log("Using the verified GitHub D2 snapshot. Drive data was left in place.")
    data = reference
    selected = verify_d2(data)
log(f"D2 data: {data}")
log(f"Verified {len(selected['outputs'])} D2 files. Splits: {', '.join(selected['splits'])}.")

Runtime: local Jupyter


Session: /tmp/retail-pilot-notebook/actual_cpu_runs/20261120T045217Z_mf_smoke_6ff46365


Source: df5e9278444acd71cca258d4ba66cb773810c180


D2 data: /tmp/retail_model_code/df5e9278444acd71cca258d4ba66cb773810c180/data/processed/d2


Verified 13 D2 files. Splits: train, validation.


## Prepare the selected Python environment

The Colab session launches the selected model in a separate Python environment. MF and LRURec use Colab's PyTorch installation. LightGBM uses the CPU package fixed in the repository. Mamba4Rec uses its own environment with the official CUDA PyTorch build and compatible native Mamba and causal-convolution wheels. For this native stack, choose a Python 3.12 runtime with a T4, L4 or A100 when available. Its first installation can take several minutes. The GPU model, driver, matching package versions and test results are recorded before training.

For local Jupyter, use the prepared model environment as the notebook kernel or set `RETAIL_MODEL_PYTHON` to its Python executable. Installing the GPU packages requires a compatible Linux/NVIDIA runtime. [PyTorch CUDA packages](https://pytorch.org/get-started/previous-versions/) · [Mamba releases](https://github.com/state-spaces/mamba/releases/tag/v2.2.5) · [causal-conv1d releases](https://github.com/Dao-AILab/causal-conv1d/releases/tag/v1.5.3.post1)

## 准备所选模型的 Python 环境

Colab 会用独立的 Python 环境启动所选模型。MF 和 LRURec 使用 Colab 提供的 PyTorch，LightGBM 使用仓库固定版本的 CPU 软件包。Mamba4Rec 在自己的环境中安装官方 CUDA 版 PyTorch 和与它兼容的 Mamba、因果卷积原生二进制包。运行这套原生依赖时，如有可选资源，选择 Python 3.12 和 T4、L4 或 A100。首次安装可能需要几分钟。GPU 型号、驱动、实际软件版本和检查结果会在训练前写入日志。

本地 Jupyter 可以使用已经装好模型依赖的环境作为内核，也可以用 `RETAIL_MODEL_PYTHON` 指定该环境的 Python 可执行文件。安装 GPU 包需要兼容的 Linux 和 NVIDIA 运行环境。[PyTorch CUDA 软件包](https://pytorch.org/get-started/previous-versions/) · [Mamba 发行版](https://github.com/state-spaces/mamba/releases/tag/v2.2.5) · [causal-conv1d 发行版](https://github.com/Dao-AILab/causal-conv1d/releases/tag/v1.5.3.post1)

In [3]:
import shutil
from urllib.request import Request, urlopen

def verify_python313_wheel(url):
    if sys.version_info[:2] == (3, 13):
        try:
            with urlopen(Request(url, method="HEAD"), timeout=30) as res:
                log(f"Verified native wheel URL (HTTP {res.status}): {url}")
        except OSError as error:
            raise RuntimeError("A required Python 3.13 wheel is unavailable. Select a Python 3.12 Colab runtime.") from error

if device == "cuda":
    if not shutil.which("nvidia-smi"):
        raise RuntimeError("Select Runtime → Change runtime type → NVIDIA GPU, reconnect, and run all cells.")
    run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv"])

if on_colab:
    if MODEL == "mamba4rec" and sys.version_info[:2] not in {(3, 11), (3, 12), (3, 13)}:
        raise RuntimeError("Select a Colab runtime with Python 3.12 to install the pinned native wheels.")
    env = Path("/content/retail_model_env") / revision / MODEL
    python = env / "bin/python"
    if not python.is_file():
        options = [] if MODEL == "mamba4rec" else ["--system-site-packages"]
        run([sys.executable, "-m", "venv", "--without-pip", *options, env])
    # Use Colab's pip only as the installer; dependencies go into the target environment.
    install = [sys.executable, "-m", "pip", "--python", python, "install", "--disable-pip-version-check", "--quiet"]
    run([*install, "-r", checkout / "requirements-models.txt"])
    if MODEL == "mamba4rec":
        if platform.machine() != "x86_64" or sys.implementation.name != "cpython":
            raise RuntimeError("The official native wheels below require x86-64 Linux CPython.")
        verify_python313_wheel("https://download-r2.pytorch.org/whl/cu124/torch-2.6.0%2Bcu124-cp313-cp313-linux_x86_64.whl")
        run([*install, "--index-url", "https://download.pytorch.org/whl/cu124", "torch==2.6.0"])
        run([*install, "transformers==4.48.3", "einops", "packaging", "ninja", "setuptools>=61"])
        abi = subprocess.check_output(
            [str(python), "-c", "import torch; print(str(torch._C._GLIBCXX_USE_CXX11_ABI).upper())"], text=True
        ).strip()
        cp = f"cp{sys.version_info.major}{sys.version_info.minor}"
        tail = f"+cu12torch2.6cxx11abi{abi}-{cp}-{cp}-linux_x86_64.whl"
        releases = [
            ("Dao-AILab/causal-conv1d", "v1.5.3.post1", "causal_conv1d-1.5.3.post1"),
            ("state-spaces/mamba", "v2.2.5", "mamba_ssm-2.2.5"),
        ]
        for repository, tag, wheel in releases:
            url = f"https://github.com/{repository}/releases/download/{tag}/{wheel}{tail}"
            verify_python313_wheel(url)
            run([*install, "--no-deps", url])
        log(f"Native wheel configuration: CUDA 12 / torch 2.6 / C++ ABI {abi} / {cp}")
else:
    python = Path(os.environ.get("RETAIL_MODEL_PYTHON", sys.executable))
log(f"Model Python: {python}")

Model Python: /workspace/online-retail-top10-recommendation/.venv/bin/python


## Check the hardware and run the native preflight

This cell uses the same Python executable and device as training. GPU verification records what the allocated Colab instance can actually execute. For Mamba4Rec, the native preflight runs the selective state-space layer, checks backward gradients, padding and checkpoint reload, and writes `mamba_preflight.json` to the session directory. A failed check stops the run at this cell. The session log keeps the original error.

## 核对硬件并运行原生算子检查

本单元格使用与训练相同的 Python 和设备，检验当前实际分配的 Colab 实例可以执行哪些操作。选择 Mamba4Rec 时，原生预检会运行选择性状态空间层，检查梯度反传、填充与模型重载，并把结果写入本次会话的 `mamba_preflight.json`。若检查失败，运行会停在这个单元格，完整报错保存在会话日志中。

In [4]:
environment_check = '''
import importlib.metadata as metadata
import json
import platform
import sys
from pathlib import Path

model, device, runtime, target = sys.argv[1:]
names = ["numpy", "pandas", "pyarrow", "scikit-learn", "lightgbm", "PyYAML"]
if model != "lightgbm":
    names.append("torch")
if model == "mamba4rec":
    names.extend(["mamba-ssm", "causal-conv1d", "transformers"])
versions = {name: metadata.version(name) for name in names}
report = {"model": model, "device": device, "runtime": runtime, "python": platform.python_version(),
          "platform": platform.platform(), "packages": versions}
if model == "lightgbm":
    import lightgbm
else:
    import torch
    if device == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("CUDA unavailable in the selected Python. Reconnect to an NVIDIA GPU runtime.")
        cuda_device = torch.cuda.current_device()
        report.update(gpu=torch.cuda.get_device_name(cuda_device), cuda=torch.version.cuda,
                      compute_capability=list(torch.cuda.get_device_capability(cuda_device)))
    probe = torch.tensor([2.0, 3.0], device=device, requires_grad=True)
    probe.square().sum().backward()
    torch.testing.assert_close(probe.grad, torch.tensor([4.0, 6.0], device=device))
    report["torch_autograd_verified"] = True
Path(target).write_text(json.dumps(report, indent=2) + chr(10), encoding="utf-8")
print(json.dumps(report, indent=2))
'''
run([python, "-c", environment_check, MODEL, device, runtime, session / "environment.json"],
    label="Checking model packages and computing on the selected device.")
if MODEL == "mamba4rec":
    run([python, "-m", "src.models.mamba4rec_adapter", "--preflight", "--output", session / "mamba_preflight.json"])

Checking model packages and computing on the selected device.


{


  "model": "mf",


  "device": "cpu",


  "runtime": "local Jupyter",


  "python": "3.12.14",


... (8 outputs left out)

    "torch": "2.8.0+cpu"


  },


  "torch_autograd_verified": true


}


## Train, score the validation catalog and reload

The command runs the repository's model entry point. **smoke** adds `--smoke`; **baseline** uses the full training and validation configuration. The run directory stores epoch or tree metrics, the selected checkpoint, Top-10 items for each validation customer query, and scores for the fixed AUC sample. Restarting this notebook creates a new run directory.

## 训练、对验证候选商品评分并重载模型

下面的命令调用仓库统一的训练入口。**smoke** 会加入 `--smoke`；**baseline** 使用完整训练与验证配置。运行目录保存每轮或每组树的指标、选定的模型文件、每个验证客户查询的 Top-10 商品，以及固定 AUC 样本的分数。重新运行整份 notebook 会新建一个结果目录。

In [5]:
command = [python, "-u", "-m", "scripts.train", "--model", MODEL, "--device", device,
           "--data-dir", data, "--output-dir", run_dir]
if RUN_SIZE == "smoke":
    command.append("--smoke")
run(command)

$ /workspace/online-retail-top10-recommendation/.venv/bin/python -u -m scripts.train --model mf --device cpu --data-dir /tmp/retail_model_code/df5e9278444acd71cca258d4ba66cb773810c180/data/processed/d2 --output-dir /tmp/retail-pilot-notebook/actual_cpu_runs/20261120T045217Z_mf_smoke_6ff46365/model --smoke


mf / cpu: 16 validation queries; /tmp/retail-pilot-notebook/actual_cpu_runs/20261120T045217Z_mf_smoke_6ff46365/model/run.json


{


  "recall_at_10": 0.011616006120316465,


  "ndcg_at_10": 0.06116342799085077,


  "supplementary_recall_at_10": 0.010322117042922964,


... (4 outputs left out)

  "auc_skipped_queries": 0,


  "auc_positive_items": 428,


  "auc_negative_items": 4280


}


## Verify the saved run and read its evidence

The final cell checks that training completed, the best model reloaded with identical Top-10 rankings, and the saved ranking and AUC outputs are readable. **Recall@10** measures how many relevant items appear in the ten recommendations. **NDCG@10** also rewards placing those items higher. **Sampled ROC AUC** compares positive scores with the protocol's fixed sampled negatives. Check the run size and number of validation queries when interpreting these metrics.

Keep the entire session folder. `session.log` captures console output, `environment.json` records the measured runtime, and `model/run.json` links the model, seed and hashes to its outputs. The last printed path is the session to open in Drive.

## 核验已保存的运行并阅读证据

最后一个单元格核对训练完成状态、最优模型重载后 Top-10 排名是否一致，以及排序和 AUC 输出能否读取。**Recall@10** 衡量十个推荐商品覆盖了多少相关商品。**NDCG@10** 还考虑相关商品的排名。**抽样 ROC AUC** 比较正例和协议中固定抽样负例的分数。解读这些指标时，请同时核对本次运行规模和验证查询数量。

保留整个会话文件夹。`session.log` 保存终端输出，`environment.json` 记录实测运行环境，`model/run.json` 记录模型、随机种子、文件校验值等信息。最后显示的路径就是本次在 Drive 中的会话目录。

In [6]:
training = json.loads((run_dir / "run.json").read_text())
if training["status"] != "complete" or training["model"] != MODEL or not training["reload_identical_top10"]:
    raise ValueError("The model did not complete its training and checkpoint reload checks.")
if training["git_commit"] != revision or training["device"] != device or training["smoke"] != (RUN_SIZE == "smoke"):
    raise ValueError("Run provenance differs from the selected source, hardware or run size.")
for name, expected in training["output_sha256"].items():
    with (run_dir / name).open("rb") as src:
        if hashlib.file_digest(src, "sha256").hexdigest() != expected:
            raise ValueError(f"Output checksum differs from run.json: {name}")
best = "best.txt" if MODEL == "lightgbm" else "best.pt"
for name in [best, "metrics.csv", "top10.parquet", "auc_scores.parquet", "query_metrics.csv"]:
    if not (run_dir / name).is_file() or (run_dir / name).stat().st_size == 0:
        raise FileNotFoundError(run_dir / name)
table_check = '''
from pathlib import Path
import pandas as pd
import sys

root = Path(sys.argv[1])
top = pd.read_parquet(root / "top10.parquet")
auc = pd.read_parquet(root / "auc_scores.parquet")
queries = pd.read_csv(root / "query_metrics.csv")
assert len(queries) > 0 and len(auc) > 0
primary = queries.loc[queries.catalog_positive_items.gt(0)]
assert set(top.query_id) == set(primary.query_id) == set(auc.query_id)
assert not top.duplicated(["query_id", "item_id"]).any()
assert top.groupby("query_id").size().eq(10).all()
print(f"Verified {len(primary):,} primary validation queries, {len(top):,} ranked items and {len(auc):,} sampled scores.")
'''
run([python, "-c", table_check, run_dir], label="Checking saved October ranking and AUC tables.")
for metric in ["recall_at_10", "ndcg_at_10", "sampled_auc"]:
    log(f"{metric}: {training['validation_metrics'][metric]}")
log(f"Top-10 identical after reload: {training['reload_identical_top10']}; maximum score change: "
    f"{training['reload_max_score_delta']}")
log(f"Session folder: {session}")

Checking saved October ranking and AUC tables.


Verified 16 primary validation queries, 160 ranked items and 4,708 sampled scores.


recall_at_10: 0.011616006120316465


ndcg_at_10: 0.06116342799085077


sampled_auc: 0.671425142751356


Top-10 identical after reload: True; maximum score change: 0.0


Session folder: /tmp/retail-pilot-notebook/actual_cpu_runs/20261120T045217Z_mf_smoke_6ff46365
